# GLips Lippenlesen mit Deep Learning

## Datenimport
Der Datensatz besteht aus Videos aus dem Hessischen Parlament wobei 500 Verschiedene Wörter enthalten sind. Die Videos sind bereits auf die Lippen gecropt und in 25 Frames pro Sekunde umgewandelt worden. Der Folder ist in Wörter unterteilt, die jeweils train test und validation Ordner enthalten.

In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.io as io

class GLipsFullClipDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, num_frames=25):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        # target number of frames per sample (videos will be truncated or padded)
        self.num_frames = num_frames
        self.samples = []

        # collect class folders
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # For each class folder, look for the requested split folder. If not found, try common alternatives.
        for cls_name in self.classes:
            split_folder = os.path.join(root_dir, cls_name, split)
            if not os.path.exists(split_folder):
                for alt in ['train', 'validation', 'val', 'test']:
                    alt_folder = os.path.join(root_dir, cls_name, alt)
                    if os.path.exists(alt_folder):
                        split_folder = alt_folder
                        break

            if os.path.exists(split_folder):
                for file in os.listdir(split_folder):
                    if file.endswith('.mp4'):
                        self.samples.append((os.path.join(split_folder, file), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        # Load entire video: try multiple backends for compatibility
        # Desired tensor shape after loading: (T, C, H, W)
        video = None

        # 1) torchvision (if available)
        if hasattr(io, 'read_video'):
            try:
                video, _, _ = io.read_video(video_path, pts_unit='sec', output_format='TCHW')
            except Exception:
                video = None

        # 2) imageio (ffmpeg) fallback
        if video is None:
            try:
                import imageio.v2 as imageio
            except Exception:
                try:
                    import imageio
                except Exception:
                    imageio = None

            if imageio is not None:
                frames = []
                try:
                    reader = imageio.get_reader(video_path, 'ffmpeg')
                    for frame in reader:
                        frames.append(frame)
                    reader.close()
                    import numpy as np
                    video_np = np.stack(frames)  # (T, H, W, C)
                    video = torch.from_numpy(video_np).permute(0, 3, 1, 2)  # (T, C, H, W)
                except Exception:
                    video = None

        # 3) OpenCV fallback
        if video is None:
            try:
                import cv2
                import numpy as np
                cap = cv2.VideoCapture(video_path)
                frames = []
                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    # convert BGR->RGB
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(frame)
                cap.release()
                if len(frames) == 0:
                    raise RuntimeError(f"No frames read from {video_path}")
                video_np = np.stack(frames)
                video = torch.from_numpy(video_np).permute(0, 3, 1, 2)
            except Exception as e:
                raise RuntimeError(
                    "Could not read video with torchvision, imageio or opencv. "
                    f"Install one of these backends. Original error: {e}"
                )

        # Convert to float in range [0,1]
        video = video.float() / 255.0

        # video currently: (T, C, H, W)
        T = video.size(0)

        # 1) If video longer than target, sample uniformly to reduce to num_frames
        if T > self.num_frames:
            import numpy as np
            indices = np.linspace(0, T - 1, num=self.num_frames).astype(int)
            video = video[indices]

        # 2) If video shorter than target, pad by repeating last frame
        elif T < self.num_frames:
            pad_count = self.num_frames - T
            last = video[-1:].repeat(pad_count, 1, 1, 1)
            video = torch.cat([video, last], dim=0)

        # Apply optional transform (expects tensor shape (T,C,H,W) or will handle accordingly)
        if self.transform:
            video = self.transform(video)

        # Returns (C, T, H, W) and label
        return video.permute(1, 0, 2, 3), label

In [3]:
# Create train and validation datasets/loaders (split folder names are tried with fallbacks inside the dataset)
train_dataset = GLipsFullClipDataset(root_dir='./GLips/lipread_files/', split='train')
val_dataset = GLipsFullClipDataset(root_dir='./GLips/lipread_files/', split='validation')

_pin = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=_pin)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=_pin)

## Modell
In README.md können Sie sich die Modellarchitektur anschauen. Das aktuelle Modell (`GLipsNet`) besteht aus einem 3D-CNN-Frontend, einem ResNet-18-Backbone, einer Projektion auf 256 Dimensionen, zwei MS-TCN-Blöcken für lokale temporale Merkmale und einem Transformer-Encoder für globale zeitliche Abhängigkeiten.

### 3D CNN + ResNet-18
Das Front-End kombiniert einen 3D-Convolutional-Layer (Kernel $5\times7\times7$) mit den Stages 1–4 des ResNet-18. Der 3D-Layer erfasst kurzzeitige Bewegungsmerkmale über die Frame-Abfolge. Die ResNet-Stages verarbeiten jeden Frame einzeln und extrahieren 512-dimensionale visuelle Merkmale. Ein adaptives Average Pooling kollabiert danach die räumlichen Dimensionen auf $1\times1$.

In [ ]:
import torch.nn as nn
import torchvision.models as models

FEAT_DIM = 512  # output channels of ResNet-18 stages 1-4


class Frontend3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        )

    def forward(self, x):
        return self.layers(x)


class ResNet2DBackend(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.layers = nn.Sequential(*list(resnet.children())[4:8])  # stages 1-4 → 512-dim

    def forward(self, x):
        return self.layers(x)


class CNN3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.frontend = Frontend3D()
        self.resnet = ResNet2DBackend()

    def forward(self, x):
        B, C, T, H, W = x.size()
        x = self.frontend(x)                               # (B, 64, T, H', W')
        x = x.transpose(1, 2).contiguous()                # (B, T, 64, H', W')
        x = x.view(-1, 64, x.size(3), x.size(4))          # (B*T, 64, H', W')
        x = self.resnet(x)                                 # (B*T, 512, H'', W'')
        return x.view(B, T, FEAT_DIM, x.size(2), x.size(3))

## MS-TCN (Multi-Scale Temporal Convolutional Network)
Zwei MS-TCN-Blöcke erfassen lokale temporale Muster auf verschiedenen Skalen. Jeder Block verwendet drei parallele Depthwise-Separable-Konvolutionen mit Kernelgrößen $k \in \{3, 5, 7\}$, die kurze, mittlere und breitere zeitliche Muster der Lippenbewegung abdecken. Die Ausgaben werden gemittelt und per Residualverbindung mit LayerNorm stabilisiert. Im Vergleich zum BiGRU ist der MS-TCN voll parallelisierbar und benötigt kein sequentielles Rollout.

In [ ]:
import torch
import torch.nn as nn

D_MODEL = 256


class MSTemporalBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        # three parallel depthwise branches capture short (k=3), mid (k=5), and broad (k=7) temporal patterns
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(d_model, d_model, k, padding=k // 2, groups=d_model),
                nn.Conv1d(d_model, d_model, 1),
                nn.BatchNorm1d(d_model),
                nn.GELU(),
            )
            for k in (3, 5, 7)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        xt = x.transpose(1, 2)
        out = sum(b(xt) for b in self.branches) / 3.0
        return self.norm(x + self.drop(out.transpose(1, 2)))

## GLipsNet – Gesamtmodell
`GLipsNet` verbindet alle Komponenten. Nach dem CNN-Frontend und der Projektion auf 256 Dimensionen verarbeiten zwei MS-TCN-Blöcke lokale temporale Muster. Ein erlernbares Positions-Embedding und 4 Transformer-Encoder-Schichten (Pre-Norm, 8 Heads, FFN-Dim 1024) modellieren globale zeitliche Abhängigkeiten. Mean Pooling über die Zeitachse aggregiert die Sequenz zu einem fixen Vektor, den eine lineare Schicht auf die 500 Zielklassen projiziert.

In [ ]:
class GLipsNet(nn.Module):
    def __init__(self, num_classes=500):
        super().__init__()
        self.cnn = CNN3D()
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Sequential(nn.Linear(FEAT_DIM, D_MODEL), nn.LayerNorm(D_MODEL))
        self.ms_tcn = nn.Sequential(MSTemporalBlock(D_MODEL), MSTemporalBlock(D_MODEL))
        self.pos_embed = nn.Parameter(torch.zeros(1, 250, D_MODEL))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=8, dim_feedforward=1024,
            dropout=0.1, activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.classifier = nn.Linear(D_MODEL, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        x = self.cnn(x)                                          # (B, T, 512, H'', W'')
        B, T, C, H, W = x.size()
        x = self.avgpool(x.view(B * T, C, H, W)).view(B, T, FEAT_DIM)  # (B, T, 512)
        x = self.proj(x)                                         # (B, T, 256)
        x = self.ms_tcn(x)                                       # (B, T, 256)
        x = x + self.pos_embed[:, :T, :]
        x = self.transformer(x)                                  # (B, T, 256)
        x = x.mean(dim=1)                                        # (B, 256)
        return self.classifier(x)                                # (B, 500)


model = GLipsNet()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
import os
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GLipsNet().to(device)

# Differential LR: pretrained ResNet backbone gets 10x lower rate
backbone_ids = {id(p) for p in model.cnn.resnet.parameters()}
param_groups = [
    {'params': [p for p in model.parameters() if id(p) not in backbone_ids], 'lr': 1e-3},
    {'params': list(model.cnn.resnet.parameters()), 'lr': 1e-4},
]
optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=(amp_dtype == torch.float16))

num_epochs = 50
warmup_epochs = 5
warmup_sched = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs])

best_val_acc = 0.0
save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True)

for epoch in tqdm(range(num_epochs), desc="Epochs"):
    model.train()
    running_loss, correct_train, num_samples = 0.0, 0, 0
    for data, target in tqdm(train_loader, desc=f"Train {epoch+1}/{num_epochs}", leave=False):
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=amp_dtype):
            logits = model(data)
            loss = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * data.size(0)
        correct_train += (logits.detach().argmax(1) == target).sum().item()
        num_samples += data.size(0)

    model.eval()
    correct1, correct5, total, val_loss = 0, 0, 0, 0.0
    with torch.no_grad():
        for data, target in tqdm(val_loader, desc="Val", leave=False):
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=amp_dtype):
                logits = model(data)
                val_loss += criterion(logits, target).item() * data.size(0)
            correct1 += (logits.argmax(1) == target).sum().item()
            correct5 += (logits.topk(5, dim=1).indices == target.unsqueeze(1)).any(1).sum().item()
            total += target.size(0)

    val_acc = correct1 / total if total else 0.0
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))
        tqdm.write(f"Epoch {epoch+1}: best val top-1={best_val_acc:.4f}, top-5={correct5/total:.4f}")

print(f"Done. Best top-1: {best_val_acc:.4f}")